# 16.7 精度能降低嗎？


同一個權重若用四bytes保存，與用兩bytes保存，前者通常能分辨更細的差異；後者省空間，也可能讓支援它的硬體更快。但兩種「兩bytes浮點數」不一定能表示相同範圍。先回顧[數值尺度](https://birdhackor.github.io/tiny-perceptron-vlm/14.2.html)：後續運算不只在乎形狀，還會受數字大小影響。本節要觀察較短的浮點表示如何改變一次矩陣乘法。

FP32是32位元浮點數，一個數字四bytes；FP16與BF16都是16位元、兩bytes。浮點表示可粗略想成「有效數字×某個次方」：存較多有效數字的位元，附近刻度更細；存較多次方的位元，可表示的極大與極小數範圍更寬。FP16通常比BF16有更細的相對精度，但最大有限值約65504；BF16保留較寬的次方範圍，最大值約3.39×10³⁸，接近FP32。

用固定輸入做CPU的BF16小實驗。第一個矩陣的數字刻意接近整數，BF16可能把1.001等值捨入到1，讓輸出有可追蹤的差異。`autocast`是PyTorch依運算選擇較低精度的介面；這裡只在包住的矩陣乘法使用，不手動改原輸入的儲存型別。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

a = torch.tensor([[1.001, 2.002], [3.003, 4.004]])
b = torch.tensor([[0.5, 1.0], [1.5, -1.0]])
full = a @ b
with torch.autocast("cpu", dtype=torch.bfloat16):
    low = a @ b
print("原輸入型別", a.dtype, "低精度輸出", low.dtype)
print("FP32", full.round(decimals=4).tolist())
print("BF16", low.float().tolist())
print("最大差異", round((full - low.float()).abs().max().item(), 4))

第一行應看到原輸入仍torch.float32，輸出是torch.bfloat16。FP32結果約 `[[3.5035,-1.001],[7.5075,-1.001]]`，BF16約 `[[3.5,-1],[7.5,-1]]`，最大差異約0.0075。`low.float()`把結果轉回FP32，只為方便比較，不能恢復先前已被捨入丟掉的細節。這個差異不是一層權重學壞了，而是表示與運算精度改變。

混合精度訓練通常讓合適的矩陣運算用低精度，某些誤差計算與累加仍用FP32。[梯度](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)是損失對參數的變化率，由反向傳播沿計算關係求出，用來決定更新方向。FP16計算中的微小梯度可能太小而無法表示，變成零，這叫下溢。GradScaler是管理這個尺度的工具：先把損失乘上倍率，例如1024，再反向傳播算出同樣放大1024倍的梯度，更新參數前再除以同一倍率，縮回原尺度。放大是為了減少計算途中小量變成零的情形，不是把學習率任意放大1024倍。這是額外的數值管理，不能只改dtype就假設訓練安全。BF16範圍較寬，也仍會有捨入誤差；每個硬體支援的運算不同，應核對結果與實際耗時。

練習只把a乘10，先預測最大絕對差異大致放大，而非保持0.0075；重跑看實際捨入結果。再把a改成全整數，預測本例兩路更接近。CPU數字核對能說明精度取捨，不能證明GPU加速或整個模型品質保持不變。
